In [ ]:
import json
import random
from collections import Counter
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score, classification_report

# reproducibility
random.seed(42)
torch.manual_seed(42)
np.random.seed(42)

## Task 1: Data Preparation

In [ ]:
LABEL_NAMES = {0: "sadness", 1: "joy", 2: "love",
               3: "anger",   4: "fear", 5: "surprise"}
NUM_CLASSES  = len(LABEL_NAMES)

# Output directory for task 1
OUTPUT_DIR = "task1_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load dataset
from datasets import load_dataset
print("Loading dataset via HuggingFace…")
raw_ds = load_dataset("dair-ai/emotion", "split")

# Extract into dictionary format
ds = {
    "train": {"text": raw_ds["train"]["text"], "label": raw_ds["train"]["label"]},
    "validation": {"text": raw_ds["validation"]["text"], "label": raw_ds["validation"]["label"]},
    "test": {"text": raw_ds["test"]["text"], "label": raw_ds["test"]["label"]}
}

print({s: len(ds[s]["text"]) for s in ds})


# STEP 1 - Class label extraction & distribution analysis
print("\n" + "="*60)
print("STEP 1 - Class Distribution")
print("="*60)

labels = {split: ds[split]["label"] for split in ("train", "validation", "test")}

def class_distribution(label_list, name):
    counts = Counter(label_list)
    total  = len(label_list)
    print(f"\n{name}  (n={total})")
    for cls_id in sorted(counts):
        pct = 100 * counts[cls_id] / total
        print(f"  {cls_id} ({LABEL_NAMES[cls_id]:8s}): {counts[cls_id]:5d}  ({pct:.1f} %)")
    return counts

train_counts = class_distribution(labels["train"],      "Train")
val_counts   = class_distribution(labels["validation"], "Validation")
test_counts  = class_distribution(labels["test"],       "Test")

# Chance accuracy & majority-class accuracy
n_train = len(labels["train"])
majority_cls  = max(train_counts, key=train_counts.get)
majority_acc  = train_counts[majority_cls] / n_train
chance_acc    = 1 / NUM_CLASSES

print(f"\nChance accuracy (uniform random):            {chance_acc:.1%}")
print(f"Majority-class accuracy (always predict "
      f"'{LABEL_NAMES[majority_cls]}'): {majority_acc:.1%}")

# Visualise class distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
fig.suptitle("Class Distribution per Split", fontsize=14, fontweight="bold")

split_data = [
    ("Train",      train_counts),
    ("Validation", val_counts),
    ("Test",       test_counts),
]
colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974", "#64B5CD"]

for ax, (split_name, counts) in zip(axes, split_data):
    cls_ids = sorted(counts)
    vals    = [counts[c] for c in cls_ids]
    bars    = ax.bar([LABEL_NAMES[c] for c in cls_ids], vals,
                     color=colors, edgecolor="white", linewidth=0.8)
    ax.set_title(split_name, fontsize=12)
    ax.set_xlabel("Emotion")
    ax.set_ylabel("Count")
    ax.tick_params(axis="x", rotation=30)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 20,
                str(v), ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "class_distribution.png"), dpi=150)
plt.close()
print("\nSaved: class_distribution.png")

# STEP 2 - Tokenisation & length statistics
print("\n" + "="*60)
print("STEP 2 - Tokenisation & Length Statistics")
print("="*60)

def tokenise(text: str):
    """
    Separates punctuation from words and splits by whitespace.
    Converts to lowercase.
    """
    # Lowercase the text
    text = text.lower()
    
    # Use regex to find all words and punctuation marks separately
    # keeps alphanumeric sequences as one token and treats non-alphanumeric/non-whitespace characters as individual tokens
    tokens = re.findall(r"[\w']+|[^\w\s]", text)
    
    return tokens

tokens = {split: [tokenise(t) for t in ds[split]["text"]]
          for split in ("train", "validation", "test")}

def length_stats(token_lists, name):
    lengths = [len(t) for t in token_lists]
    print(f"\n{name}")
    print(f"  Range : {min(lengths)} - {max(lengths)} tokens")
    print(f"  Mean  : {np.mean(lengths):.2f}")
    print(f"  Std   : {np.std(lengths):.2f}")
    print(f"  Median: {np.median(lengths):.1f}")
    print(f"  95th %-ile: {np.percentile(lengths, 95):.1f}")
    return lengths

train_lengths = length_stats(tokens["train"],      "Train")
val_lengths   = length_stats(tokens["validation"], "Validation")
test_lengths  = length_stats(tokens["test"],       "Test")

# Visualise length distributions 
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
fig.suptitle("Token-Length Distribution per Split", fontsize=14, fontweight="bold")

for ax, (split_name, lengths) in zip(axes, [
        ("Train",      train_lengths),
        ("Validation", val_lengths),
        ("Test",       test_lengths)]):
    ax.hist(lengths, bins=30, color="#4C72B0", edgecolor="white", alpha=0.85)
    ax.axvline(np.mean(lengths),   color="red",    linestyle="--", label=f"mean={np.mean(lengths):.1f}")
    ax.axvline(np.percentile(lengths, 95), color="orange", linestyle=":", label="95th pct")
    ax.set_title(split_name)
    ax.set_xlabel("Tokens")
    ax.set_ylabel("Count")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "length_distribution.png"), dpi=150)
plt.close()
print("\nSaved: length_distribution.png")

# STEP 3 - Vocabulary construction
print("\n" + "="*60)
print("STEP 3 - Vocabulary Construction")
print("="*60)

# Special tokens
PAD_TOKEN = "<PAD>"   # index 0 - used for padding
UNK_TOKEN = "<UNK>"   # index 1 - used for out-of-vocabulary words

# Build vocab from train split only
all_train_tokens = [tok for seq in tokens["train"] for tok in seq]
token_freq = Counter(all_train_tokens)

# Sort by frequency (descending) for a clean vocab
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
for token, _ in token_freq.most_common():
    vocab[token] = len(vocab)

print(f"Vocabulary size (incl. special tokens): {len(vocab):,}")
print(f"  <PAD> index : {vocab[PAD_TOKEN]}")
print(f"  <UNK> index : {vocab[UNK_TOKEN]}")
print(f"  Most common token: '{token_freq.most_common(1)[0][0]}' "
      f"({token_freq.most_common(1)[0][1]:,} occurrences)")

# STEP 4 - Encoding texts
print("\n" + "="*60)
print("STEP 4 - Encoding Texts (with OOV → <UNK>)")
print("="*60)

def encode(token_list, vocab, unk_idx):
    return [vocab.get(tok, unk_idx) for tok in token_list]

unk_idx = vocab[UNK_TOKEN]

encoded = {split: [encode(seq, vocab, unk_idx) for seq in tokens[split]]
           for split in ("train", "validation", "test")}

# OOV rate check
for split in ("train", "validation", "test"):
    all_ids = [idx for seq in encoded[split] for idx in seq]
    oov_rate = sum(1 for i in all_ids if i == unk_idx) / len(all_ids)
    print(f"  {split:12s} OOV rate: {oov_rate:.2%}")

# STEP 5 - Padding / truncation → Tensors + DataLoaders
print("\n" + "="*60)
print("STEP 5 - Padding, Tensors & DataLoaders")
print("="*60)

# Chosen MAX_LEN based on length distribution
MAX_LEN   = int(np.percentile(train_lengths, 95))
print(f"Chosen MAX_LEN (95th percentile of train): {MAX_LEN}")

pad_idx = vocab[PAD_TOKEN]

def pad_or_truncate(encoded_seq, max_len, pad_idx):
    """Truncate to max_len, then right-pad with pad_idx."""
    seq = encoded_seq[:max_len]                  # truncate
    seq = seq + [pad_idx] * (max_len - len(seq)) # pad
    return seq

def build_tensor(encoded_seqs, label_list, max_len, pad_idx):
    padded  = [pad_or_truncate(s, max_len, pad_idx) for s in encoded_seqs]
    X = torch.tensor(padded,     dtype=torch.long)
    y = torch.tensor(label_list, dtype=torch.long)
    return X, y

X_train, y_train = build_tensor(encoded["train"],      labels["train"],      MAX_LEN, pad_idx)
X_val,   y_val   = build_tensor(encoded["validation"], labels["validation"], MAX_LEN, pad_idx)
X_test,  y_test  = build_tensor(encoded["test"],       labels["test"],       MAX_LEN, pad_idx)

print(f"\nTensor shapes:")
print(f"  X_train: {X_train.shape}   y_train: {y_train.shape}")
print(f"  X_val  : {X_val.shape}   y_val  : {y_val.shape}")
print(f"  X_test : {X_test.shape}   y_test : {y_test.shape}")

# DataLoaders
BATCH_SIZE = 64

train_loader = DataLoader(TensorDataset(X_train, y_train),
                          batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(TensorDataset(X_val,   y_val),
                          batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(TensorDataset(X_test,  y_test),
                          batch_size=BATCH_SIZE, shuffle=False)

print(f"\nDataLoaders (batch_size={BATCH_SIZE}):")
print(f"  Train batches      : {len(train_loader)}")
print(f"  Validation batches : {len(val_loader)}")
print(f"  Test batches       : {len(test_loader)}")

# Sanity-check: inspect one batch
xb, yb = next(iter(train_loader))
print(f"\nSample batch — X: {xb.shape}, y: {yb.shape}")
print(f"  First sequence (first 20 ids): {xb[0, :20].tolist()}")
print(f"  Label: {yb[0].item()} ({LABEL_NAMES[yb[0].item()]})")

# Summary figure - padding illustration
fig, ax = plt.subplots(figsize=(10, 3))
sample_lens = [len(s) for s in encoded["train"][:200]]
sample_pads = [MAX_LEN - min(l, MAX_LEN) for l in sample_lens]
x = np.arange(200)
ax.bar(x, [min(l, MAX_LEN) for l in sample_lens], label="tokens",  color="#4C72B0")
ax.bar(x, sample_pads, bottom=[min(l, MAX_LEN) for l in sample_lens],
       label="padding", color="#DDDDDD", alpha=0.7)
ax.axhline(MAX_LEN, color="red", linestyle="--", label=f"MAX_LEN={MAX_LEN}")
ax.set_xlabel("Example index (first 200 train examples)")
ax.set_ylabel("Sequence length")
ax.set_title("Padding Illustration (first 200 training examples)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "padding_illustration.png"), dpi=150)
plt.close()
print("\nSaved: padding_illustration.png")

# Save artefacts
torch.save({
    "vocab":        vocab,
    "pad_idx":      pad_idx,
    "unk_idx":      unk_idx,
    "max_len":      MAX_LEN,
    "X_train": X_train, "y_train": y_train,
    "X_val":   X_val,   "y_val":   y_val,
    "X_test":  X_test,  "y_test":  y_test,
}, os.path.join(OUTPUT_DIR, "task1_artefacts.pt"))

print(f"\nAll artefacts saved to {OUTPUT_DIR}")
print("\n✓  Task 1 complete.")

In [ ]:
# Class weights used in the final submitted workflow
train_counts = Counter(labels["train"])
counts = [train_counts[i] for i in range(NUM_CLASSES)]
total = sum(counts)

class_weights = torch.tensor(
    [total / (NUM_CLASSES * count) for count in counts],
    dtype=torch.float,
)

print("Class weights:", class_weights.tolist())

## Task 2: Model 1 - Baseline Feedforward Neural Network (MLP)

In [ ]:
# Output directory for Task 2
OUTPUT_DIR = "task2_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load task 1 artefacts
data = torch.load("task1_results/task1_artefacts.pt")
vocab_size = len(data["vocab"])
pad_idx = data["pad_idx"]

class EmotionMLP(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.5) 

    def forward(self, x):
        embedded = self.embedding(x)
        pooled = torch.mean(embedded, dim=1) 
        x = self.dropout(self.relu(self.fc1(pooled)))
        x = self.dropout(self.relu(self.fc2(x)))
        return self.fc3(x)

In [ ]:
# 1. Model Initialization
EMBEDDING_DIM = 128
HIDDEN_DIM = 128 
LR = 0.001
WEIGHT_DECAY = 1e-4 # L2 regularization
PATIENCE = 5 # Number of epochs to wait before stopping
EPOCHS = 30

# Initialize the model
model = EmotionMLP(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, 6, pad_idx).to('cpu')
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# Scheduler: Reduces LR by 0.1x if val_loss doesn't improve for 2 epochs
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=1)

# Variables for early stopping tracking
best_val_loss = float('inf')
epochs_without_improvement = 0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}

# 2. Training Loop 
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for texts, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    avg_train_loss = train_loss / len(train_loader)
    
    # Validation phase
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for texts, labels in val_loader:
            outputs = model(texts)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    avg_val_loss = val_loss / len(val_loader)
    val_acc = 100 * correct / total
    
    # 3. Update Scheduler & Early Stopping Logic
    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_acc'].append(val_acc)
    
    scheduler.step(avg_val_loss)
    
    print(f"Epoch [{epoch+1}] - Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    # Check for improvement
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        epochs_without_improvement = 0
        # Save the best version of the model
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model1.pt"))
    else:
        epochs_without_improvement += 1
        
    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping triggered! No improvement for {PATIENCE} epochs.")
        break

# Load the best state found during training before final test
model.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_model1.pt")))

# 4. Final Evaluation & Visualization

# Loss and accuracy curves
plt.figure(figsize=(12, 5))

# Plot training and validation loss
plt.subplot(1, 2, 1)
plt.plot(history['train_loss'], label='Train Loss', color='#4C72B0')
plt.plot(history['val_loss'], label='Val Loss', color='#C44E52')
plt.title('Training and Validation Loss (MLP)')
plt.xlabel('Epochs')
plt.ylabel('Cross-Entropy Loss')
plt.legend()

# Plot validation accuracy
plt.subplot(1, 2, 2)
plt.plot(history['val_acc'], label='Val Accuracy', color='#55A868')
plt.title('Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy (%)')
plt.legend()

plt.tight_layout()
# Save the figure
plt.savefig(os.path.join(OUTPUT_DIR, "model1_curves.png"), dpi=150)
plt.show()

# Final evaluation on the test set
model.eval()
test_correct, test_total = 0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    for texts, labels in test_loader:
        outputs = model(texts)
        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        all_preds.extend(predicted.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

test_acc = 100 * test_correct / test_total
macro_f1 = f1_score(all_labels, all_preds, average="macro")

print("\n" + "=" * 30)
print("FINAL TEST EVALUATION")
print("=" * 30)
print(f"Final Test Accuracy: {test_acc:.2f}%")
print(f"Macro F1: {macro_f1:.4f}")
print(classification_report(
    all_labels,
    all_preds,
    target_names=list(LABEL_NAMES.values()),
))

## Task 3: Model 2 - Transformer Classifier

In [ ]:
OUTPUT_DIR = "task3_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load task 1 artefacts (same data pipeline as Model 1)
data = torch.load("task1_results/task1_artefacts.pt")
vocab_size = len(data["vocab"])
pad_idx = data["pad_idx"]
MAX_LEN = data["max_len"]

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE}")

# Building block 1 - Multi-Head Self-Attention
class MultiHeadSelfAttention(nn.Module):

    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, "embed_dim must be divisible by num_heads"

        self.num_heads = num_heads
        self.head_dim  = embed_dim // num_heads
        self.scale     = self.head_dim ** -0.5   # 1/sqrt(d_k)

        # Fused Q, K, V projections for efficiency
        self.qkv_proj = nn.Linear(embed_dim, 3 * embed_dim)
        self.out_proj  = nn.Linear(embed_dim, embed_dim)
        self.attn_drop = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):

        B, T, E = x.shape
        H, D = self.num_heads, self.head_dim

        # Project and split into H heads
        qkv = self.qkv_proj(x)                   
        q, k, v = qkv.chunk(3, dim=-1)             
        q = q.view(B, T, H, D).transpose(1, 2)       
        k = k.view(B, T, H, D).transpose(1, 2)
        v = v.view(B, T, H, D).transpose(1, 2)

        # Scaled dot-product attention scores
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale  

        # Mask padding positions (set to -inf so softmax → 0)
        if key_padding_mask is not None:
            scores = scores.masked_fill(
                key_padding_mask[:, None, None, :], float("-inf")
            )

        attn_weights = torch.softmax(scores, dim=-1)  
        attn_weights = self.attn_drop(attn_weights)

        # Weighted sum of values
        out = torch.matmul(attn_weights, v)       
        out = out.transpose(1, 2).contiguous()       
        out = out.view(B, T, E)                      
        return self.out_proj(out)                     

# Building block 2 - Transformer Block
class TransformerBlock(nn.Module):

    def __init__(self, embed_dim: int, num_heads: int,
                 ffn_dim: int, dropout: float = 0.1):
        super().__init__()

        # Sub-layer 1: Multi-Head Self-Attention
        self.norm1  = nn.LayerNorm(embed_dim)
        self.attn   = MultiHeadSelfAttention(embed_dim, num_heads, dropout)
        self.drop1  = nn.Dropout(dropout)

        # Sub-layer 2: Position-wise Feed-Forward Network
        self.norm2  = nn.LayerNorm(embed_dim)
        self.ffn    = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, embed_dim),
        )
        self.drop2  = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        # Residual connection 1: attention
        x = x + self.drop1(self.attn(self.norm1(x), key_padding_mask))
        # Residual connection 2: FFN
        x = x + self.drop2(self.ffn(self.norm2(x)))
        return x


# Full model - EmotionTransformer
class EmotionTransformer(nn.Module):

    def __init__(self, vocab_size: int, embed_dim: int, num_heads: int,
                 ffn_dim: int, num_layers: int, num_classes: int,
                 max_len: int, pad_idx: int, dropout: float = 0.1):
        super().__init__()
        self.pad_idx = pad_idx

        # Token embedding 
        self.token_emb = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        # Learnable positional encoding
        self.pos_emb   = nn.Embedding(max_len, embed_dim)
        self.emb_drop  = nn.Dropout(dropout)

        # Stack of Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ffn_dim, dropout)
            for _ in range(num_layers)
        ])

        # Final LayerNorm 
        self.norm = nn.LayerNorm(embed_dim)

        # Classification head
        self.head_drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(embed_dim, num_classes)

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)
                if m.padding_idx is not None:
                    m.weight.data[m.padding_idx].zero_()

    def forward(self, x):

        B, T = x.shape
        device = x.device

        # Build key-padding mask: True where token == <PAD>
        pad_mask = (x == self.pad_idx)              

        # Embeddings = token + positional
        positions = torch.arange(T, device=device).unsqueeze(0) 
        emb = self.token_emb(x) + self.pos_emb(positions)      
        emb = self.emb_drop(emb)

        # Pass through all Transformer blocks
        h = emb
        for block in self.blocks:
            h = block(h, key_padding_mask=pad_mask)
        h = self.norm(h)                       

        # Mean pooling - average over real (non-padding) tokens only
        real_mask = (~pad_mask).float().unsqueeze(-1)  # (B, T, 1)
        pooled = (h * real_mask).sum(dim=1) / real_mask.sum(dim=1).clamp(min=1e-9)

        # Classification head
        logits = self.classifier(self.head_drop(pooled)) 
        return logits


### 3A: Baseline Transformer

In [ ]:
# Hyperparameters
EMBED_DIM   = 128      # embedding / model dimension
NUM_HEADS   = 4        # number of attention heads (128 / 4 = 32-dim per head)
FFN_DIM     = 256      # inner dimension of the position-wise FFN
NUM_LAYERS  = 2        # number of stacked Transformer blocks
DROPOUT     = 0.1
LR          = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE    = 5
EPOCHS      = 30

# Instantiate model 
model2 = EmotionTransformer(
    vocab_size  = vocab_size,
    embed_dim   = EMBED_DIM,
    num_heads   = NUM_HEADS,
    ffn_dim     = FFN_DIM,
    num_layers  = NUM_LAYERS,
    num_classes = 6,
    max_len     = MAX_LEN,
    pad_idx     = pad_idx,
    dropout     = DROPOUT,
).to(DEVICE)

total_params = sum(p.numel() for p in model2.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

criterion2 = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
optimizer2 = optim.AdamW(model2.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler2 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer2, mode="min", factor=0.1, patience=1
)

# Training loop 
best_val_loss2          = float("inf")
epochs_without_improve2 = 0
history2 = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS):
    # Train 
    model2.train()
    train_loss = 0.0
    for texts, labels in train_loader:
        texts, labels = texts.to(DEVICE), labels.to(DEVICE)
        optimizer2.zero_grad()
        outputs = model2(texts)
        loss = criterion2(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model2.parameters(), max_norm=1.0)  # gradient clipping
        optimizer2.step()
        train_loss += loss.item()
    avg_train_loss = train_loss / len(train_loader)

    # Validate 
    model2.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for texts, labels in val_loader:
            texts, labels = texts.to(DEVICE), labels.to(DEVICE)
            outputs = model2(texts)
            val_loss += criterion2(outputs, labels).item()
            _, predicted = torch.max(outputs, 1)
            total   += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_val_loss = val_loss / len(val_loader)
    val_acc      = 100 * correct / total

    history2["train_loss"].append(avg_train_loss)
    history2["val_loss"].append(avg_val_loss)
    history2["val_acc"].append(val_acc)

    scheduler2.step(avg_val_loss)

    print(f"Epoch [{epoch+1:2d}] - "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc: {val_acc:.2f}%")

    # Early stopping
    if avg_val_loss < best_val_loss2:
        best_val_loss2          = avg_val_loss
        epochs_without_improve2 = 0
        torch.save(model2.state_dict(),
                   os.path.join(OUTPUT_DIR, "best_model2.pt"))
    else:
        epochs_without_improve2 += 1
        if epochs_without_improve2 >= PATIENCE:
            print(f"Early stopping triggered after {epoch+1} epochs.")
            break

# Reload best checkpoint
model2.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_model2.pt"),
                                   map_location=DEVICE))


# Evaluation & Visualisation

# Loss / accuracy curves
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history2["train_loss"], label="Train Loss",  color="#4C72B0")
plt.plot(history2["val_loss"],   label="Val Loss",    color="#C44E52")
plt.title("Training and Validation Loss (Transformer)")
plt.xlabel("Epochs")
plt.ylabel("Cross-Entropy Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history2["val_acc"], label="Val Accuracy", color="#55A868")
plt.title("Validation Accuracy (Transformer)")
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model2_curves.png"), dpi=150)
plt.show()

# Final test evaluation
model2.eval()
test_correct, test_total = 0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    for texts, labels in test_loader:
        texts, labels = texts.to(DEVICE), labels.to(DEVICE)
        outputs = model2(texts)
        _, predicted = torch.max(outputs, 1)

        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        all_preds.extend(predicted.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

test_acc2 = 100 * test_correct / test_total
macro_f1 = f1_score(all_labels, all_preds, average="macro")

print("\n" + "=" * 30)
print("FINAL TEST EVALUATION (Transformer)")
print("=" * 30)
print(f"Final Test Accuracy: {test_acc2:.2f}%")
print(f"Macro F1: {macro_f1:.4f}")
print(classification_report(
    all_labels,
    all_preds,
    target_names=list(LABEL_NAMES.values()),
))

### 3B: Hyperparameter Tuning and Final Transformer

In [ ]:
import itertools, time, math

OUTPUT_DIR = "task3_results_tuning"
os.makedirs(OUTPUT_DIR, exist_ok=True)

data = torch.load("task1_results/task1_artefacts.pt")
vocab_size = len(data["vocab"])
pad_idx = data["pad_idx"]
MAX_LEN = data["max_len"]

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Device: {DEVICE}")

# Fixed across all configs
EMBED_DIM  = 128
NUM_HEADS  = 4
TUNE_EPOCHS = 15
TUNE_PATIENCE = 3

# Search space
PARAM_GRID = {
    "dropout"      : [0.1, 0.3, 0.4],
    "weight_decay" : [1e-4, 1e-3],
    "num_layers"   : [1, 2],
    "ffn_dim"      : [128, 256],
    "lr"           : [1e-3, 5e-4],
}


# Warmup + cosine-decay scheduler 
def make_scheduler(optimizer, warmup_epochs, total_epochs):

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs         
        progress = (epoch - warmup_epochs) / max(1, total_epochs - warmup_epochs)
        return max(1e-6, 0.5 * (1.0 + math.cos(math.pi * progress)))  # cosine
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# Helper: train one fresh model, return best val loss & acc 
def run_config(cfg, epochs=TUNE_EPOCHS, patience=TUNE_PATIENCE, seed=42):

    torch.manual_seed(seed)

    model = EmotionTransformer(
        vocab_size  = vocab_size,
        embed_dim   = EMBED_DIM,
        num_heads   = NUM_HEADS,
        ffn_dim     = cfg["ffn_dim"],
        num_layers  = cfg["num_layers"],
        num_classes = 6,
        max_len     = MAX_LEN,
        pad_idx     = pad_idx,
        dropout     = cfg["dropout"],
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    optimizer = optim.AdamW(model.parameters(),
                            lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = make_scheduler(optimizer, warmup_epochs=2, total_epochs=epochs)

    best_val_loss = float("inf")
    best_val_acc  = 0.0
    no_improve    = 0

    for epoch in range(epochs):
        model.train()
        for texts, lbls in train_loader:
            texts, lbls = texts.to(DEVICE), lbls.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(texts), lbls)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for texts, lbls in val_loader:
                texts, lbls = texts.to(DEVICE), lbls.to(DEVICE)
                out = model(texts)
                val_loss += criterion(out, lbls).item()
                correct  += (out.argmax(1) == lbls).sum().item()
                total    += lbls.size(0)

        avg_val_loss = val_loss / len(val_loader)
        val_acc      = 100 * correct / total

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_val_acc  = val_acc
            no_improve    = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                break

    return best_val_loss, best_val_acc


# Grid search 
keys    = list(PARAM_GRID.keys())
values  = list(PARAM_GRID.values())
configs = [dict(zip(keys, combo)) for combo in itertools.product(*values)]

print(f"\nRunning grid search over {len(configs)} configurations…\n")
print(f"{'#':>3}  {'dropout':>7}  {'wd':>6}  {'layers':>6}  {'ffn':>5}  {'lr':>6}  "
      f"{'val_loss':>9}  {'val_acc':>8}")
print("-" * 70)

results = []
for i, cfg in enumerate(configs):
    t0 = time.time()
    val_loss, val_acc = run_config(cfg)
    elapsed = time.time() - t0
    results.append({**cfg, "val_loss": val_loss, "val_acc": val_acc})
    print(f"{i+1:>3}  {cfg['dropout']:>7.2f}  {cfg['weight_decay']:>6.0e}  "
          f"{cfg['num_layers']:>6}  {cfg['ffn_dim']:>5}  {cfg['lr']:>6.0e}  "
          f"{val_loss:>9.4f}  {val_acc:>7.2f}%  ({elapsed:.0f}s)")

# Best config
best = min(results, key=lambda r: r["val_loss"])
print("\n" + "="*70)
print("BEST CONFIGURATION (lowest val loss):")
for k, v in best.items():
    print(f"  {k:15s}: {v}")
print("="*70)

# Visualise: mean val loss per hyperparameter value 
import pandas as pd

df = pd.DataFrame(results)

fig, axes = plt.subplots(1, len(keys), figsize=(4 * len(keys), 4), sharey=False)
fig.suptitle("Grid Search: Mean Val Loss by Hyperparameter", fontsize=13, fontweight="bold")

for ax, key in zip(axes, keys):
    grouped = df.groupby(key)["val_loss"].mean().reset_index()
    ax.bar(grouped[key].astype(str), grouped["val_loss"],
           color="#4C72B0", edgecolor="white", alpha=0.85)
    ax.set_title(key, fontsize=10)
    ax.set_xlabel(key)
    ax.set_ylabel("Mean Val Loss" if key == keys[0] else "")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "hyperparam_search.png"), dpi=150)
plt.show()
print("Saved: hyperparam_search.png")

BEST_CFG = best
print("\nBEST_CFG stored - run the next cell to train the final model.")

In [ ]:
# Final Model 2 - fresh model trained with best hyperparameters
# Uses the same warmup + cosine-decay schedule as the grid search runs

torch.manual_seed(42)

EPOCHS_FINAL  = 30
PATIENCE_FINAL = 5

# Fresh model 
model2 = EmotionTransformer(
    vocab_size  = vocab_size,
    embed_dim   = EMBED_DIM,
    num_heads   = NUM_HEADS,
    ffn_dim     = BEST_CFG["ffn_dim"],
    num_layers  = BEST_CFG["num_layers"],
    num_classes = 6,
    max_len     = MAX_LEN,
    pad_idx     = pad_idx,
    dropout     = BEST_CFG["dropout"],
).to(DEVICE)

total_params = sum(p.numel() for p in model2.parameters() if p.requires_grad)
print(f"Final model — trainable parameters: {total_params:,}")
print(f"Config: {BEST_CFG}\n")

criterion2 = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
optimizer2 = optim.AdamW(model2.parameters(),
                         lr=BEST_CFG["lr"],
                         weight_decay=BEST_CFG["weight_decay"])
scheduler2 = make_scheduler(optimizer2, warmup_epochs=2, total_epochs=EPOCHS_FINAL)

best_val_loss2          = float("inf")
epochs_without_improve2 = 0
history2 = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS_FINAL):
    # Train
    model2.train()
    train_loss = 0.0
    for texts, lbls in train_loader:
        texts, lbls = texts.to(DEVICE), lbls.to(DEVICE)
        optimizer2.zero_grad()
        loss = criterion2(model2(texts), lbls)
        loss.backward()
        nn.utils.clip_grad_norm_(model2.parameters(), 1.0)
        optimizer2.step()
        train_loss += loss.item()
    scheduler2.step()
    avg_train_loss = train_loss / len(train_loader)

    # Validate
    model2.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for texts, lbls in val_loader:
            texts, lbls = texts.to(DEVICE), lbls.to(DEVICE)
            out = model2(texts)
            val_loss += criterion2(out, lbls).item()
            correct  += (out.argmax(1) == lbls).sum().item()
            total    += lbls.size(0)

    avg_val_loss = val_loss / len(val_loader)
    val_acc      = 100 * correct / total

    history2["train_loss"].append(avg_train_loss)
    history2["val_loss"].append(avg_val_loss)
    history2["val_acc"].append(val_acc)

    print(f"Epoch [{epoch+1:2d}] - "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc: {val_acc:.2f}%")

    if avg_val_loss < best_val_loss2:
        best_val_loss2          = avg_val_loss
        epochs_without_improve2 = 0
        torch.save(model2.state_dict(),
                   os.path.join(OUTPUT_DIR, "best_model2.pt"))
    else:
        epochs_without_improve2 += 1
        if epochs_without_improve2 >= PATIENCE_FINAL:
            print(f"Early stopping after epoch {epoch+1}.")
            break

# Reload best checkpoint
model2.load_state_dict(torch.load(os.path.join(OUTPUT_DIR, "best_model2.pt"),
                                   map_location=DEVICE))

# Loss / accuracy curves 
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history2["train_loss"], label="Train Loss",  color="#4C72B0")
plt.plot(history2["val_loss"],   label="Val Loss",    color="#C44E52")
plt.title("Training and Validation Loss (Transformer - tuned)")
plt.xlabel("Epochs")
plt.ylabel("Cross-Entropy Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history2["val_acc"], label="Val Accuracy", color="#55A868")
plt.title("Validation Accuracy (Transformer - tuned)")
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "model2_tuned_curves.png"), dpi=150)
plt.show()

# Final test evaluation
model2.eval()
test_correct, test_total = 0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    for texts, lbls in test_loader:
        texts, lbls = texts.to(DEVICE), lbls.to(DEVICE)
        out = model2(texts)
        preds = out.argmax(dim=1)

        test_correct += (preds == lbls).sum().item()
        test_total += lbls.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(lbls.cpu().tolist())

test_acc2 = 100 * test_correct / test_total
macro_f1 = f1_score(all_labels, all_preds, average="macro")

print("\n" + "=" * 45)
print("FINAL TEST EVALUATION (Transformer - tuned)")
print("=" * 45)
print(f"Best hyperparameters : {BEST_CFG}")
print(f"Final Test Accuracy  : {test_acc2:.2f}%")
print(f"Macro F1             : {macro_f1:.4f}")
print(classification_report(
    all_labels,
    all_preds,
    target_names=list(LABEL_NAMES.values()),
))

## Task 4: Analysis of Tuned Transformer Classifier Model

### 4A: Basic Analysis - Confusion Matrix and Misclassified Examples

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import pandas as pd

# 1. Generate Predictions for the Test Set 
model2.eval()
all_preds = []
all_labels = []
test_texts_raw = raw_ds["test"]["text"] # Get original text for analysis

with torch.no_grad():
    for texts, labels in test_loader:
        texts = texts.to(DEVICE)
        outputs = model2(texts)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# 2. Confusion Matrix 
cm = confusion_matrix(all_labels, all_preds)
cm_df = pd.DataFrame(cm, index=[LABEL_NAMES[i] for i in range(6)], 
                         columns=[LABEL_NAMES[i] for i in range(6)])

plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix: Transformer Classifier')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.savefig(os.path.join(OUTPUT_DIR, "confusion_matrix.png"))
plt.show()

# 3. Error Analysis (Finding Misclassified Examples) 
print("\n" + "="*60)
print("ERROR ANALYSIS: MISCLASSIFIED EXAMPLES")
print("="*60)

shown_errors = set()

for i in range(len(all_labels)):
    actual_idx = all_labels[i]
    # If the prediction is WRONG and we haven't shown an error for this actual class yet
    if all_preds[i] != actual_idx and actual_idx not in shown_errors:
        print(f"Text: {test_texts_raw[i]}")
        print(f"  -> Actual Emotion:    {LABEL_NAMES[actual_idx]}")
        print(f"  -> Model's Guess:     {LABEL_NAMES[all_preds[i]]}")
        print("-" * 30)
        
        shown_errors.add(actual_idx)
        
    if len(shown_errors) >= 6: # Stop once we have one error for each class
        break

# 4. Success Analysis (Finding Correctly Classified Examples) 
print("\n" + "="*60)
print("SUCCESS ANALYSIS: CORRECTLY CLASSIFIED EXAMPLES")
print("="*60)

shown_emotions = set()

for i in range(len(all_labels)):
    actual_idx = all_labels[i]
    # If the prediction is correct AND we haven't shown this class yet
    if all_preds[i] == actual_idx and actual_idx not in shown_emotions:
        print(f"Text: {test_texts_raw[i]}")
        print(f"  -> Predicted: {LABEL_NAMES[actual_idx]}")
        print(f"  -> Actual:    {LABEL_NAMES[actual_idx]}")
        print("-" * 30)
        
        shown_emotions.add(actual_idx)
        
    if len(shown_emotions) >= 6: # Stop once we have one of each class
        break

### 4B: Interactive Testing Code

In [ ]:
def predict_emotion(user_input, actual_label):
    # 1. Preprocess (Tokenize -> Encode -> Pad)
    tokens = tokenise(user_input)
    encoded = [data["vocab"].get(t, data["unk_idx"]) for t in tokens]
    padded = encoded[:data["max_len"]] + [data["pad_idx"]] * (
        data["max_len"] - len(encoded)
    )

    # 2. Predict
    model2.eval()
    input_tensor = torch.LongTensor([padded]).to(DEVICE)

    with torch.no_grad():
        logits = model2(input_tensor)
        prediction_idx = torch.argmax(logits, dim=1).item()

    predicted_label = LABEL_NAMES[prediction_idx]

    # 3. Output
    print(f"Text: {user_input}")
    print(f"Actual Label: {actual_label}")
    print(f"Predicted Label: {predicted_label}")

    # 4. Quick Check
    if actual_label == predicted_label:
        print("✅ Match!")
    else:
        print("❌ Misclassified")

In [ ]:
predict_emotion(
    user_input="i am not happy",
    actual_label="sadness"
)

## Task 5: Fine-tuning a Pre-trained Language Model (DistilBERT)

In [ ]:
# Task 5 - Fine-tuning DistilBERT for Emotion Classification
# Model choice: distilbert-base-uncased
# A compact pretrained Transformer used as the comparison model.
# It uses its own WordPiece tokenizer and is fine-tuned for emotion classification.

from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    DistilBertConfig,
    get_linear_schedule_with_warmup,
)
from torch.utils.data import Dataset, DataLoader
import torch, torch.nn as nn, torch.optim as optim
import numpy as np, matplotlib.pyplot as plt, os

OUTPUT_DIR = "task5_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
print(f"Device: {DEVICE}")

LABEL_NAMES = {0: "sadness", 1: "joy", 2: "love",
               3: "anger",   4: "fear", 5: "surprise"}

# STEP 1 - Load raw texts from the Hugging Face dataset
# We go back to raw text (not token ids) because DistilBERT uses its own
# WordPiece tokenizer — our custom vocabulary is incompatible.
from datasets import load_dataset
print("Loading dataset…")
raw_ds = load_dataset("dair-ai/emotion", "split")

train_texts  = raw_ds["train"]["text"]
train_labels = raw_ds["train"]["label"]
val_texts    = raw_ds["validation"]["text"]
val_labels   = raw_ds["validation"]["label"]
test_texts   = raw_ds["test"]["text"]
test_labels  = raw_ds["test"]["label"]

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")

# STEP 2 - DistilBERT tokenizer
TOKENIZER_NAME = "distilbert-base-uncased"
tokenizer = DistilBertTokenizerFast.from_pretrained(TOKENIZER_NAME)

# MAX_LEN: DistilBERT supports up to 512; 64 covers 99%+ of this dataset
# (avg sentence ~19 tokens) and keeps training fast
MAX_LEN_BERT = 64
print(f"\nTokenizer loaded. Max sequence length set to: {MAX_LEN_BERT}")

# Quick sanity check
sample = tokenizer(train_texts[0], truncation=True,
                   max_length=MAX_LEN_BERT, padding="max_length")
print(f"Sample input_ids length : {len(sample['input_ids'])}")
print(f"Original text           : {train_texts[0][:80]}")

# STEP 3 - PyTorch Dataset wrapper
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            list(texts),
            truncation=True,
            max_length=max_len,
            padding="max_length",
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids"      : self.encodings["input_ids"][idx],
            "attention_mask" : self.encodings["attention_mask"][idx],
            "labels"         : self.labels[idx],
        }

print("\nTokenising splits…")
train_dataset = EmotionDataset(train_texts,  train_labels, tokenizer, MAX_LEN_BERT)
val_dataset   = EmotionDataset(val_texts,    val_labels,   tokenizer, MAX_LEN_BERT)
test_dataset  = EmotionDataset(test_texts,   test_labels,  tokenizer, MAX_LEN_BERT)

BATCH_SIZE = 32
train_loader_bert = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader_bert   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader_bert  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders ready (batch_size={BATCH_SIZE})")
print(f"  Train batches: {len(train_loader_bert)} | "
      f"Val batches: {len(val_loader_bert)} | "
      f"Test batches: {len(test_loader_bert)}")


In [ ]:
# STEP 4 - Hyperparameter tuning (learning rate & dropout)

import itertools, time

TUNE_GRID = {
    "lr"      : [2e-5, 3e-5, 5e-5],
    "dropout" : [0.1, 0.2, 0.3],
}
TUNE_EPOCHS   = 3
TUNE_WARMUP   = 0.1   # fraction of steps used for warmup

def run_bert_config(lr, dropout, epochs=TUNE_EPOCHS, seed=42):
    torch.manual_seed(seed)

    config = DistilBertConfig.from_pretrained("distilbert-base-uncased")
    config.dropout = dropout
    config.attention_dropout = dropout
    config.seq_classif_dropout = dropout
    config.num_labels = NUM_CLASSES

    model = DistilBertForSequenceClassification.from_pretrained(
        "distilbert-base-uncased",
        config=config,
    ).to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    total_steps  = len(train_loader_bert) * epochs
    warmup_steps = int(total_steps * TUNE_WARMUP)
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    best_val_loss = float("inf")
    best_val_acc  = 0.0

    for epoch in range(epochs):
        model.train()
        for batch in train_loader_bert:
            input_ids = batch["input_ids"].to(DEVICE)
            attn_mask = batch["attention_mask"].to(DEVICE)
            labels    = batch["labels"].to(DEVICE)
            optimizer.zero_grad()
            out  = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            out.loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for batch in val_loader_bert:
                input_ids = batch["input_ids"].to(DEVICE)
                attn_mask = batch["attention_mask"].to(DEVICE)
                labels    = batch["labels"].to(DEVICE)
                out = model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
                val_loss += out.loss.item()
                preds     = out.logits.argmax(dim=1)
                correct  += (preds == labels).sum().item()
                total    += labels.size(0)

        avg_val_loss = val_loss / len(val_loader_bert)
        val_acc      = 100 * correct / total

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_val_acc  = val_acc

    return best_val_loss, best_val_acc

# Run grid
keys    = list(TUNE_GRID.keys())
configs = [dict(zip(keys, c)) for c in itertools.product(*TUNE_GRID.values())]

print(f"Grid search over {len(configs)} configs ({TUNE_EPOCHS} epochs each)…\n")
print(f"{'#':>3}  {'lr':>6}  {'dropout':>7}  {'val_loss':>9}  {'val_acc':>8}")
print("-" * 45)

bert_results = []
for i, cfg in enumerate(configs):
    t0 = time.time()
    vl, va = run_bert_config(**cfg)
    bert_results.append({**cfg, "val_loss": vl, "val_acc": va})
    print(f"{i+1:>3}  {cfg['lr']:>6.0e}  {cfg['dropout']:>7.2f}  "
          f"{vl:>9.4f}  {va:>7.2f}%  ({time.time()-t0:.0f}s)")

BEST_BERT_CFG = min(bert_results, key=lambda r: r["val_loss"])
print("\n" + "="*50)
print("BEST CONFIG:")
for k, v in BEST_BERT_CFG.items():
    print(f"  {k:12s}: {v}")
print("="*50)


In [ ]:
# STEP 5 - Final fine-tuning run with best hyperparameters

torch.manual_seed(42)

EPOCHS_FINAL  = 5
PATIENCE_FINAL = 2
WARMUP_FRAC   = 0.1

config = DistilBertConfig.from_pretrained("distilbert-base-uncased")
config.dropout = BEST_BERT_CFG["dropout"]
config.attention_dropout = BEST_BERT_CFG["dropout"]
config.seq_classif_dropout = BEST_BERT_CFG["dropout"]
config.num_labels = NUM_CLASSES

bert_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    config=config,
).to(DEVICE)

total_params = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
print(f"Trainable parameters : {total_params:,}")
print(f"Best config          : lr={BEST_BERT_CFG['lr']:.0e}, dropout={BEST_BERT_CFG['dropout']}\n")

optimizer_bert = optim.AdamW(bert_model.parameters(),
                              lr=BEST_BERT_CFG["lr"], weight_decay=1e-2)
total_steps  = len(train_loader_bert) * EPOCHS_FINAL
warmup_steps = int(total_steps * WARMUP_FRAC)
scheduler_bert = get_linear_schedule_with_warmup(
    optimizer_bert,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

best_val_loss_bert = float("inf")
no_improve_bert    = 0
history_bert = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(EPOCHS_FINAL):
    # Train
    bert_model.train()
    train_loss = 0.0
    for batch in train_loader_bert:
        input_ids = batch["input_ids"].to(DEVICE)
        attn_mask = batch["attention_mask"].to(DEVICE)
        labels    = batch["labels"].to(DEVICE)
        optimizer_bert.zero_grad()
        out = bert_model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
        out.loss.backward()
        nn.utils.clip_grad_norm_(bert_model.parameters(), 1.0)
        optimizer_bert.step()
        scheduler_bert.step()
        train_loss += out.loss.item()
    avg_train_loss = train_loss / len(train_loader_bert)

    # Validate
    bert_model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for batch in val_loader_bert:
            input_ids = batch["input_ids"].to(DEVICE)
            attn_mask = batch["attention_mask"].to(DEVICE)
            labels    = batch["labels"].to(DEVICE)
            out = bert_model(input_ids=input_ids, attention_mask=attn_mask, labels=labels)
            val_loss += out.loss.item()
            preds     = out.logits.argmax(dim=1)
            correct  += (preds == labels).sum().item()
            total    += labels.size(0)

    avg_val_loss = val_loss / len(val_loader_bert)
    val_acc      = 100 * correct / total

    history_bert["train_loss"].append(avg_train_loss)
    history_bert["val_loss"].append(avg_val_loss)
    history_bert["val_acc"].append(val_acc)

    print(f"Epoch [{epoch+1:2d}] - Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    if avg_val_loss < best_val_loss_bert:
        best_val_loss_bert = avg_val_loss
        no_improve_bert    = 0
        torch.save(bert_model.state_dict(),
                   os.path.join(OUTPUT_DIR, "best_bert_model.pt"))
    else:
        no_improve_bert += 1
        if no_improve_bert >= PATIENCE_FINAL:
            print(f"Early stopping after epoch {epoch+1}.")
            break

# Reload best checkpoint
bert_model.load_state_dict(torch.load(
    os.path.join(OUTPUT_DIR, "best_bert_model.pt"), map_location=DEVICE))

# Curves 
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(history_bert["train_loss"], label="Train Loss", color="#4C72B0")
plt.plot(history_bert["val_loss"],   label="Val Loss",   color="#C44E52")
plt.title("Training and Validation Loss (DistilBERT)")
plt.xlabel("Epochs"); plt.ylabel("Cross-Entropy Loss"); plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_bert["val_acc"], label="Val Accuracy", color="#55A868")
plt.title("Validation Accuracy (DistilBERT)")
plt.xlabel("Epochs"); plt.ylabel("Accuracy (%)"); plt.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "bert_curves.png"), dpi=150)
plt.show()

# Test evaluation 
bert_model.eval()
test_correct, test_total = 0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_loader_bert:
        input_ids = batch["input_ids"].to(DEVICE)
        attn_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        out = bert_model(input_ids=input_ids, attention_mask=attn_mask)
        preds = out.logits.argmax(dim=1)

        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

test_acc_bert = 100 * test_correct / test_total
macro_f1 = f1_score(all_labels, all_preds, average="macro")

print("\n" + "=" * 45)
print("FINAL TEST EVALUATION (DistilBERT fine-tuned)")
print("=" * 45)
print(f"Final Test Accuracy: {test_acc_bert:.2f}%")
print(f"Macro F1: {macro_f1:.4f}")
print(classification_report(
    all_labels,
    all_preds,
    target_names=list(LABEL_NAMES.values()),
))

In [ ]:
# STEP 6 - On-the-fly prediction + failure case analysis
# (mirrors Task 4 for direct comparison with scratch-trained models)

LABEL_NAMES = {0: "sadness", 1: "joy", 2: "love",
               3: "anger",   4: "fear", 5: "surprise"}

def predict_bert(text):
    """Classify a raw text string with the fine-tuned DistilBERT model."""
    bert_model.eval()
    enc = tokenizer(
        text, truncation=True, max_length=MAX_LEN_BERT,
        padding="max_length", return_tensors="pt"
    )
    input_ids = enc["input_ids"].to(DEVICE)
    attn_mask = enc["attention_mask"].to(DEVICE)
    with torch.no_grad():
        logits = bert_model(input_ids=input_ids, attention_mask=attn_mask).logits
    probs     = torch.softmax(logits, dim=1).squeeze().cpu().tolist()
    pred_idx  = int(torch.argmax(logits, dim=1).item())
    return LABEL_NAMES[pred_idx], pred_idx, probs

# Failure case inspection 
print("=" * 60)
print("FAILURE CASE ANALYSIS")
print("=" * 60)

# Collect misclassified examples from the test set
bert_model.eval()
failures = []   # (text, true_label, pred_label)

with torch.no_grad():
    idx = 0
    for batch in test_loader_bert:
        input_ids = batch["input_ids"].to(DEVICE)
        attn_mask = batch["attention_mask"].to(DEVICE)
        labels    = batch["labels"].to(DEVICE)
        preds     = bert_model(input_ids=input_ids, attention_mask=attn_mask).logits.argmax(1)
        for j in range(len(labels)):
            text = test_texts[idx]
            true_l, pred_l = labels[j].item(), preds[j].item()
            if true_l != pred_l and len(failures) < 10:
                failures.append((text, true_l, pred_l))
            idx += 1

print(f"\nFirst 10 misclassified examples:")
print("-" * 60)
for text, true_l, pred_l in failures:
    print(f"  Text : {text}")
    print(f"  True : {LABEL_NAMES[true_l]:10s}  Predicted: {LABEL_NAMES[pred_l]}")
    print()

# On-the-fly prediction demo
print("=" * 60)
print("ON-THE-FLY PREDICTION DEMO")
print("=" * 60)

demo_texts = [
    # Correct predictions (intuitive)
    "i feel so happy and grateful today",
    "i am absolutely devastated by the news",
    # Ambiguous / boundary cases
    "i feel a little strange about all of this",
    "my heart is racing and i cannot calm down",
    # Longer text (generalisation test)
    ("i woke up this morning feeling completely lost, "
     "unsure of what i want and where i am going, "
     "and the sadness just kept washing over me"),
    # Minimal modification pair (correct → misclassified)
    "i feel so loved and cared for",          # expect: love
    "i feel so watched and cared for",        # swapping 'loved' → 'watched'
]

print()
for text in demo_texts:
    label, idx, probs = predict_bert(text)
    top2 = sorted(enumerate(probs), key=lambda x: -x[1])[:2]
    top2_str = ", ".join(f"{LABEL_NAMES[i]}={p:.2f}" for i, p in top2)
    print(f"  Text      : {text[:70]}")
    print(f"  Predicted : {label:10s}  (top-2: {top2_str})")
    print()
